# 09 — Gate 2b: MGA reference run (spec v0.10 §7; estimator `mga_maxham_v1`)

The re-operationalized within-formulation estimator on the reference formulation
(`s0_ssp585_theta5`): certified **anchor** (with the t=0 tails-equivalence assert), then
**k=50 maximally-diverse members** of the g-band at **g = 5%**, plus **f(g) probes at 2% and
10%** — each g on a fresh copy of the compiled model with one appended band-wall row
(objective ≤ (1+g)·z*). Direct Gurobi calls (`mga_core.R`, toy-verified); ~1 min/iterate ⇒
**~2.5–3 h total**; resumable per g; **live internet** (WLS) throughout. Run 08 (the S4 pilot;
it writes `scenarios_v2.json`) first. Kernel: `R (y2y)`.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx   <- pr_setup(mpath, PROJ)

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y


In [2]:
# ---- ingest (v0.11 critical path: the ORIGINAL 8-feature stack) ----------------------------
ctx <- modifyList(ctx, pr_ingest(ctx))
stopifnot("expected 8 continuous features -- if the tail contingency fired, revisit this notebook deliberately" =
            ctx$n_cont == 8)
ctx <- modifyList(ctx, pr_planning_units(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [3]:
# ---- S0 from scenarios_v2 (weights unchanged; tail targets 0.0 = mathematically absent) ----
sc  <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/scenarios_v2.json"))
S0W <- sc$S0_balanced$weights
S0T <- sc$S0_balanced$targets
cat(sprintf("scenarios_v2 (%s, estimator %s): S0 targets:\n",
            sc$`_meta`$spec_version, sc$`_meta`$estimator))
for (nm in names(S0T)) cat(sprintf("  %-32s %.3f\n", nm, as.numeric(S0T[[nm]])))

ctx <- pr_override(ctx, targets = S0T, feature_weight_multipliers = S0W,
                   results_subdir = "iter10_y2y_s0_mga")   # scratch tag; outputs go to runs/
ctx <- modifyList(ctx, pr_weights(ctx))
ctx <- modifyList(ctx, pr_targets(ctx))
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params
cm <- mga_compile(ctx)

scenarios_v2 (v0.11, estimator mga_maxham_v1): S0 targets:
  irrecoverable_carbon_m_soc       0.332
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.46, transboundary_connectivity=0.6693, climate_corridors=1.1714, irrecoverable_carbon_m_soc=0.4646, irrecoverable_carbon_biomass=0.1986, aoh_richness_birds=1.3286, aoh_richness_mammals=1.7075
  override results_subdir   -> iter10_y2y_s0_mga
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: climate_type_macrorefugia=1.46, transboundary_connectivity=0.6693, climate_corridors=1.1714, irrecoverable_carbon_m_soc=0.4646, irrecoverable_carbon_biomass=0.1986, aoh_richness_birds=1.3286, aoh_richness_mammals=1.7075
  outputs  -> output_data/iter10_y2y_s0_mga
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight climate_type_macrorefugia x1.5 -> 1.4600
  up-weight transboundary_connectivity x0.7 -> 0.6693
  up-weig

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.7075)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272962 cols (1272914 pu + 48 aux) x 49 rows | 191029 locked pu | modelsense min


In [4]:
# ---- anchor: certified optimum of the v2-stack S0 (t=0 tails must change NOTHING) ----------
anchor <- mga_anchor(cm, opt_gap = 1e-4)
Z_ITER9 <- 5.362813   # pool-best certified optimum on the v1 stack (results_log R6.2)
stopifnot("anchor deviates from iter9 -- t=0 tails are NOT inert; STOP" =
            abs(anchor$z - Z_ITER9) < 1e-3)
cat(sprintf("t=0 equivalence PROVEN: anchor %.6f vs iter9 %.6f (delta %.1e)\n",
            anchor$z, Z_ITER9, abs(anchor$z - Z_ITER9)))

OUT <- file.path(PROJ, "analyses/y2y/runs/s0_ssp585_theta5")
dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
# anchor raster + meta
r <- terra::rast(ctx$cost); v <- rep(NA_integer_, terra::ncell(r))
v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
terra::writeRaster(r, file.path(OUT, "anchor.tif"), overwrite = TRUE, datatype = "INT1U",
                   NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
jsonlite::write_json(list(
  formulation_id = "s0_ssp585_theta5", estimator = "mga_maxham_v1",
  anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
  anchor_runtime_s = anchor$runtime, iter9_reference = Z_ITER9,
  k = 50, g_levels = c(0.02, 0.05, 0.10), opt_gap = 1e-4,
  mip_gap_dist = 0.01, time_limit_iter = 900,
  scenarios_meta = sc$`_meta`, created_utc = format(Sys.time(), tz = "UTC")),
  file.path(OUT, "gate2b_meta.json"), auto_unbox = TRUE, pretty = TRUE)
cat("anchor.tif + gate2b_meta.json written\n")

anchor: objective 5.362840 (bound 5.362840, gap 0.00e+00) | 381,874 selected | 10 s
t=0 equivalence PROVEN: anchor 5.362840 vs iter9 5.362813 (delta 2.7e-05)
anchor.tif + gate2b_meta.json written


In [5]:
# ---- MGA sweep 1/3: g = 5%, k = 50 (the headline band; ~50 min) ----------------------------
if (file.exists(file.path(OUT, "mga_g05.tif"))) {
  cat("g05 already generated -- skipped\n")
} else {
  gen05 <- mga_generate(cm, anchor, g = 0.05, k = 50)
  mga_write(gen05, cm, ctx$cost, OUT, "g05")
}

band wall appended: obj0 . x <= 5.630982  (g = 0.05 on z* = 5.362840)
g=0.05 iter 01/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 358,905 | 10 s
g=0.05 iter 02/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 318,984 | 8 s
g=0.05 iter 03/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 268,198 | 13 s
g=0.05 iter 04/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 216,794 | 7 s
g=0.05 iter 05/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 180,372 | 8 s
g=0.05 iter 06/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 294,066 | 14 s
g=0.05 iter 07/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 271,674 | 16 s
g=0.05 iter 08/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 242,440 | 15 s
g=0.05 iter 09/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 215,428 | 16 s
g=0.05 iter 10/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 262,198 | 12 s
g=0.05 iter 11/50: band 5.630982 (+5.00% of z*) OK | ham(anchor) 266,660 | 14 s
g=0.05 iter 12/50: band 5.630982 (+5.00% of z*) OK | 

In [6]:
# ---- MGA sweep 2/3: g = 2% probe (~50 min) -------------------------------------------------
if (file.exists(file.path(OUT, "mga_g02.tif"))) {
  cat("g02 already generated -- skipped\n")
} else {
  gen02 <- mga_generate(cm, anchor, g = 0.02, k = 50)
  mga_write(gen02, cm, ctx$cost, OUT, "g02")
}

band wall appended: obj0 . x <= 5.470097  (g = 0.02 on z* = 5.362840)
g=0.02 iter 01/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 260,238 | 13 s
g=0.02 iter 02/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 186,364 | 12 s
g=0.02 iter 03/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 135,424 | 17 s
g=0.02 iter 04/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 171,082 | 16 s
g=0.02 iter 05/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 179,246 | 14 s
g=0.02 iter 06/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 153,664 | 8 s
g=0.02 iter 07/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 164,716 | 10 s
g=0.02 iter 08/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 176,866 | 10 s
g=0.02 iter 09/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 159,403 | 9 s
g=0.02 iter 10/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 161,691 | 9 s
g=0.02 iter 11/50: band 5.470097 (+2.00% of z*) OK | ham(anchor) 175,728 | 10 s
g=0.02 iter 12/50: band 5.470097 (+2.00% of z*) OK | 

In [7]:
# ---- MGA sweep 3/3: g = 10% probe (~50 min) ------------------------------------------------
if (file.exists(file.path(OUT, "mga_g10.tif"))) {
  cat("g10 already generated -- skipped\n")
} else {
  gen10 <- mga_generate(cm, anchor, g = 0.10, k = 50)
  mga_write(gen10, cm, ctx$cost, OUT, "g10")
}

band wall appended: obj0 . x <= 5.899125  (g = 0.1 on z* = 5.362840)
g=0.1 iter 01/50: band 5.899124 (+10.00% of z*) OK | ham(anchor) 381,691 | 9 s
g=0.1 iter 02/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 381,660 | 11 s
g=0.1 iter 03/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 376,213 | 8 s
g=0.1 iter 04/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 354,180 | 8 s
g=0.1 iter 05/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 289,653 | 11 s
g=0.1 iter 06/50: band 5.899124 (+10.00% of z*) OK | ham(anchor) 270,992 | 7 s
g=0.1 iter 07/50: band 5.899124 (+10.00% of z*) OK | ham(anchor) 283,424 | 9 s
g=0.1 iter 08/50: band 5.899124 (+10.00% of z*) OK | ham(anchor) 320,986 | 9 s
g=0.1 iter 09/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 371,667 | 7 s
g=0.1 iter 10/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 343,427 | 7 s
g=0.1 iter 11/50: band 5.899125 (+10.00% of z*) OK | ham(anchor) 205,847 | 33 s
g=0.1 iter 12/50: band 5.899124 (+10.00% of z*) OK | ham(an

In [8]:
# ---- run summary ---------------------------------------------------------------------------
for (tag in c("g02", "g05", "g10")) {
  f <- file.path(OUT, sprintf("certificates_%s.csv", tag))
  if (!file.exists(f)) { cat(sprintf("%s: not run\n", tag)); next }
  d <- read.csv(f)
  cat(sprintf("%s: %d members | band all OK: %s | ham(anchor) %s-%s | %d dup | %d time-limited | %.1f min\n",
              tag, nrow(d), all(d$band_ok),
              format(min(d$hamming_to_anchor), big.mark = ","),
              format(max(d$hamming_to_anchor), big.mark = ","),
              sum(d$duplicate), sum(d$status == "TIME_LIMIT"), sum(d$runtime_s) / 60))
}
cat("\nnext: analyses/y2y/10_gate2b_analysis.ipynb (kernel y2y-geo)\n")

g02: 50 members | band all OK: TRUE | ham(anchor) 135,424-260,238 | 0 dup | 0 time-limited | 10.0 min
g05: 50 members | band all OK: TRUE | ham(anchor) 180,372-358,905 | 0 dup | 0 time-limited | 13.3 min
g10: 50 members | band all OK: TRUE | ham(anchor) 121,302-381,691 | 0 dup | 0 time-limited | 8.4 min

next: analyses/y2y/10_gate2b_analysis.ipynb (kernel y2y-geo)
